# 看流式执行的第二组API
astream_events  
固定的api用法，本事是异步的，用于获取图运行过程中产生的Runnable标准事件，有很多东西， 生命周期，父子调用关系，输入和输出和元数据， 展示内容多，封装完整

底层使用runnable的接口，runnable接口在langgraph和langchain里面都比较广泛，基本上能够使用invoke或者stream，基本都有runnable的接口

用的工具、模型组件基本都是runnable组件

都是runnable组件，会产生3个runnable事件
1. 开始阶段 start
2. 中间 sream
3. 结束 end

类型比较多，有chain, chat_model, llm , tool多种类型


```json
{
    'event': 'on_chain_start',
    'data': {
        'input': {
            'initial_state': '初始状态' # 输入
        }
    },
    'name': 'LangGraph',
    'tags': [],
    'run_id': '019eeedd-9dfd-7400-b465-6b95f7292333',
    'metadata': {},
    'parent_ids': []
}
```

例子，链条开始的位置  

具体打印的astream事件什么状态，还要看使用的什么版本
可以打印v1,v2,v3， 但是langgraph1.1.2，只支持v1/v2版本，默认v2

看下v2就行，区别就是封装的内容有一点点不一样，现在的v2版本还是标准的runnable事件 ，parent_ids可以表达完整的父级运行链结构， v1版本没有parnetnts_id，看父子关系方便

v3是全新的一套协议， 有个类型化 投影消费有个表格，有不同的点，能够调用不同的消息， 状态快照，计算图的最终输出等等，打印后能够看出最后的内容，不同投影可以独立或并发消费，就是一下子可以并行消费很多种，功能更强大，区分开了

这里主要展示v2版本，复制一下代码就行， 和之前没太大区别，就是串行的节点，打印一下

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class OverAllState(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str

def node_a(state: OverAllState) -> OverAllState:
    return {
        "node_a_output": "节点A的输出"
    }

def node_b(state: OverAllState) -> OverAllState:
    return {
        "node_b_output": "节点B的输出"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
async for chunk in graph.astream_events( # 之前是graph.stream，这里直接astream_events就行
    {"initial_state": "初始状态"},
    version="v2"
):
    print(chunk)

{'event': 'on_chain_start', 'data': {'input': {'initial_state': '初始状态'}}, 'name': 'LangGraph', 'tags': [], 'run_id': '01a0b3d5-2f17-7621-99f2-eadcd60d2aee', 'metadata': {}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'initial_state': '初始状态'}}, 'name': 'node_a', 'tags': ['graph:step:1'], 'run_id': '01a0b3d5-2f1b-7ee1-91aa-957b66b5a50d', 'metadata': {'langgraph_step': 1, 'langgraph_node': 'node_a', 'langgraph_triggers': ('branch:to:node_a',), 'langgraph_path': ('__pregel_pull', 'node_a'), 'langgraph_checkpoint_ns': 'node_a:c84f8df8-391c-e66d-3b1d-7eaae2e1dc63'}, 'parent_ids': ['01a0b3d5-2f17-7621-99f2-eadcd60d2aee']}
{'event': 'on_chain_stream', 'run_id': '01a0b3d5-2f1b-7ee1-91aa-957b66b5a50d', 'name': 'node_a', 'tags': ['graph:step:1'], 'metadata': {'langgraph_step': 1, 'langgraph_node': 'node_a', 'langgraph_triggers': ('branch:to:node_a',), 'langgraph_path': ('__pregel_pull', 'node_a'), 'langgraph_checkpoint_ns': 'node_a:c84f8df8-391c-e66d-3b1d-7eaae2e1dc63'}, 'dat

最开始 on_chain_start 初始状态，刚开始执行的位置， data是输入的数据，name node_a， 就是node_a执行之前

{
    'event': 'on_chain_start', # 整个langgraph开始之前
    'data': {
        'input': {
            'initial_state': '初始状态'
        }
    },
    'name': 'LangGraph',
    'tags': [],
    'run_id': '019ef40f-2522-7212-9ad6-cb83c499652d',
    'metadata': {},
    'parent_ids': []
}

{
    'event': 'on_chain_start', # node_a开始之前
    'data': {
        'input': {
            'initial_state': '初始状态'
        }
    },
    'name': 'node_a',
    'tags': [
        'graph:step:1' # 步数1
    ],
    'run_id': '019ef40f-2524-7f83-9501-9248f93fd0a4',
    'metadata': {
        'langgraph_step': 1,
        'langgraph_node': 'node_a', #名称node_a
        'langgraph_triggers': (
            'branch:to:node_a',  # 由于开始位置传给node_a
        ),
        'langgraph_path': (
            '__pregel_pull',
            'node_a'
        ),
        'langgraph_checkpoint_ns': 'node_a:9a47e3e1-ccab-85cc-f837-f3c986bafd1f'
    },
    'parent_ids': [
        '019ef40f-2522-7212-9ad6-cb83c499652d'   # 链式结构
    ]
}

{
    'event': 'on_chain_stream', # 执行汇总
    'run_id': '019ef40f-2524-7f83-9501-9248f93fd0a4',
    'name': 'node_a',  # node_a执行中， node_a执行中，还是第一步
    'tags': [
        'graph:step:1'
    ],
    'metadata': {
        'langgraph_step': 1,
        'langgraph_node': 'node_a',
        'langgraph_triggers': (
            'branch:to:node_a',
        ),
        'langgraph_path': (
            '__pregel_pull',
            'node_a'
        ),
        'langgraph_checkpoint_ns': 'node_a:9a47e3e1-ccab-85cc-f837-f3c986bafd1f'
    },
    'data': {
        'chunk': {
            'node_a_output': '节点A的输出' # 会有数据的输出
        }
    },
    'parent_ids': [
        '019ef40f-2522-7212-9ad6-cb83c499652d'
    ]
}

{
    'event': 'on_chain_end', # node_a执行完毕
    'data': {  # 执行完成， 把数据合并起来
        'output': {
            'node_a_output': '节点A的输出'
        },
        'input': {
            'initial_state': '初始状态'
        }
    },
    'run_id': '019ef40f-2524-7f83-9501-9248f93fd0a4',
    'name': 'node_a',
    'tags': [
        'graph:step:1'
    ],
    'metadata': {
        'langgraph_step': 1,
        'langgraph_node': 'node_a',
        'langgraph_triggers': (
            'branch:to:node_a',
        ),
        'langgraph_path': (
            '__pregel_pull',
            'node_a'
        ),
        'langgraph_checkpoint_ns': 'node_a:9a47e3e1-ccab-85cc-f837-f3c986bafd1f'
    },
    'parent_ids': [
        '019ef40f-2522-7212-9ad6-cb83c499652d'
    ]
}

{
    'event': 'on_chain_stream', #再a的执行 ，中间点
    'run_id': '019ef40f-2522-7212-9ad6-cb83c499652d',
    'name': 'LangGraph',
    'tags': [],
    'metadata': {},
    'data': {
        'chunk': {
            'node_a': {
                'node_a_output': '节点A的输出'
            }
        }
    },
    'parent_ids': []
}

{
    'event': 'on_chain_start',  #b开始
    'data': {
        'input': {
            'initial_state': '初始状态',
            'node_a_output': '节点A的输出'
        }
    },
    'name': 'node_b',
    'tags': [
        'graph:step:2'
    ],
    'run_id': '019ef40f-2526-70a3-8ad1-04ce18598b65',
    'metadata': {
        'langgraph_step': 2,
        'langgraph_node': 'node_b',
        'langgraph_triggers': (
            'branch:to:node_b',
        ),
        'langgraph_path': (
            '__pregel_pull',
            'node_b'
        ),
        'langgraph_checkpoint_ns': 'node_b:1fc56ad3-e5a9-e2a2-4c3e-b8cffec34d9c'
    },
    'parent_ids': [
        '019ef40f-2522-7212-9ad6-cb83c499652d'
    ]
}

{
    'event': 'on_chain_stream',
    'run_id': '019ef40f-2526-70a3-8ad1-04ce18598b65',
    'name': 'node_b',
    'tags': [
        'graph:step:2'
    ],
    'metadata': {
        'langgraph_step': 2,
        'langgraph_node': 'node_b',
        'langgraph_triggers': (
            'branch:to:node_b',
        ),
        'langgraph_path': (
            '__pregel_pull',
            'node_b'
        ),
        'langgraph_checkpoint_ns': 'node_b:1fc56ad3-e5a9-e2a2-4c3e-b8cffec34d9c'
    },
    'data': {
        'chunk': {
            'node_b_output': '节点B的输出'
        }
    },
    'parent_ids': [
        '019ef40f-2522-7212-9ad6-cb83c499652d'
    ]
}

{
    'event': 'on_chain_end',
    'data': {
        'output': { # 合并
            'node_b_output': '节点B的输出'
        },
        'input': {
            'initial_state': '初始状态',
            'node_a_output': '节点A的输出'
        }
    },
    'run_id': '019ef40f-2526-70a3-8ad1-04ce18598b65',
    'name': 'node_b',
    'tags': [
        'graph:step:2'
    ],
    'metadata': {
        'langgraph_step': 2,
        'langgraph_node': 'node_b',
        'langgraph_triggers': (
            'branch:to:node_b',
        ),
        'langgraph_path': (
            '__pregel_pull',
            'node_b'
        ),
        'langgraph_checkpoint_ns': 'node_b:1fc56ad3-e5a9-e2a2-4c3e-b8cffec34d9c'
    },
    'parent_ids': [
        '019ef40f-2522-7212-9ad6-cb83c499652d'
    ]
}

{
    'event': 'on_chain_stream', # 两个节点间的中间状态
    'run_id': '019ef40f-2522-7212-9ad6-cb83c499652d',
    'name': 'LangGraph',
    'tags': [],
    'metadata': {},
    'data': {
        'chunk': {
            'node_b': {
                'node_b_output': '节点B的输出'
            }
        }
    },
    'parent_ids': []
}

{
    'event': 'on_chain_end', # 整个执行完毕，数据合并并输出， 如果需要有这样日志输出打印，就可以用astream_events，不需要就没必要打印
    'data': {
        'output': {
            'initial_state': '初始状态',
            'node_a_output': '节点A的输出',
            'node_b_output': '节点B的输出'
        }
    },
    'run_id': '019ef40f-2522-7212-9ad6-cb83c499652d',
    'name': 'LangGraph',
    'tags': [],
    'metadata': {},
    'parent_ids': []
}

**事件分类**：纵向按生命周期分为 `on_xxx_start`（开始）、`on_xxx_stream`（中间结果）、`on_xxx_end`（结束）；横向按 `Runnable` 类型分为以下事件：

| 组件类型 | 事件前缀 | name 示例 |
|:---|:---|:---|
| 模型 | `on_chat_model_*` / `on_llm_*` | `'[model name]'` |
| 工具 | `on_tool_*`（仅 start/end） | `'some_tool'` |
| 检索器 | `on_retriever_*` | `'[retriever name]'` |
| 提示词模板 | `on_prompt_*` | `'[template_name]'` |
| 通用（含编译图、普通节点） | `on_chain_*` | `'format_docs'` 等 |


如果是其他组件会有其他名称，比如on_tool_start


**关键字段**：

| 字段 | 含义 |
|:---|:---|
| `event` | 事件类型 |
| `run_id` | 当前 Runnable 运行实例的唯一 ID |
| `name` | 运行实例名称 |
| `metadata` | 运行相关元数据 |
| `data` | 事件输入/输出/中间结果 |
| `parent_ids` | 从根运行实例到父运行实例的 ID 链（最外层 LangGraph 为空，子节点包含父 |

字段的含义

调试可以用一下，但是大多情况没有这么复杂，也不一定看到这么底层的内容，如果有这方面需要，就可以把stream_events节点，每个调用打印一下看下一下去查找bug，输出调试就介绍完了
